# Nang Cao: Sensitivity Analysis + Feature Engineering Nang Cao + Time-Based Validation

**Da cap nhat theo `../01-tong-quan/lo-trinh-theo-de-cuong-giang-vien.md` (Buoc 4-6)** -- sua 5 diem con thieu:

1. Them dac trung `FrequencyTrend` va `RepeatPurchaseRatio` (Buoc 4)
2. Merge `Country_grouped` (da encode) vao ma tran dac trung modeling (Buoc 4)
3. Them **LightGBM** lam mo hinh thu 4 (Buoc 5)
4. Chuyen tu StratifiedKFold ngau nhien sang **Time-based Walk-Forward Validation** dua tren cac moc cutoff (Buoc 5)
5. Bo sung **PR-AUC** ben canh ROC-AUC (Buoc 6)

**Truoc khi chay**: dat file `online_retail_II.csv` vao thu muc `data/` cung cap voi notebook nay.

---
# Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import wilcoxon
import xgboost as xgb
import lightgbm as lgb

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)


In [ ]:
df_raw = pd.read_csv('data/online_retail_II.csv', encoding='ISO-8859-1')
df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'])
df_raw['is_cancelled'] = df_raw['Invoice'].astype(str).str.startswith('C')

def lam_sach_du_lieu(df_input):
    """Giong het ham da dung o eda-va-tien-xu-ly-du-lieu.ipynb (Phan B, Buoc 11)."""
    df = df_input.copy()
    df = df[~df['Invoice'].astype(str).str.startswith('C')]
    non_product_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'ADJUST', 'ADJUST2', 'CRUK']
    df = df[~df['StockCode'].astype(str).isin(non_product_codes)]
    df = df.dropna(subset=['Customer ID'])
    df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
    df = df.drop_duplicates()
    df['Revenue'] = df['Quantity'] * df['Price']
    return df.reset_index(drop=True)

df_clean = lam_sach_du_lieu(df_raw)
max_date = df_clean['InvoiceDate'].max()
min_date = df_clean['InvoiceDate'].min()
print(f"Du lieu tu {min_date.date()} den {max_date.date()}")
print(f"Sau lam sach: {len(df_clean):,} dong, {df_clean['Customer ID'].nunique():,} khach hang")


---
# PHAN 1 -- Sensitivity Analysis (Nhieu moc Cutoff)

**Tai sao lam**: dinh nghia churn (3/6 thang) chi dang tin cay neu ty le churn KHONG thay doi qua manh khi doi moc cutoff. Danh sach cutoff hop le tu phan nay se duoc TAI SU DUNG lam co so cho Time-based Walk-Forward Validation o Phan 3.

In [ ]:
def kiem_tra_va_gan_nhan(df_clean_input, cutoff_date, churn_window_days, max_date_input):
    """Tra ve (labels_series, is_valid). is_valid=False neu bi right-censoring."""
    required_end = cutoff_date + pd.Timedelta(days=churn_window_days)
    if required_end > max_date_input:
        return None, False

    obs = df_clean_input[df_clean_input['InvoiceDate'] < cutoff_date]
    future = df_clean_input[(df_clean_input['InvoiceDate'] >= cutoff_date) & (df_clean_input['InvoiceDate'] < required_end)]

    customers_before = obs['Customer ID'].unique()
    if len(customers_before) == 0:
        return None, False

    active_future = set(future['Customer ID'].unique())
    labels = pd.Series(
        [0 if cid in active_future else 1 for cid in customers_before],
        index=customers_before, name='churn'
    )
    return labels, True


In [ ]:
cutoff_candidates = pd.date_range(
    start=min_date + pd.DateOffset(months=8),
    end=max_date - pd.DateOffset(months=7),
    freq='2MS'
)
print(f"So moc cutoff se thu: {len(cutoff_candidates)}")

# Loc lai chi giu cac cutoff HOP LE (khong bi right-censoring) cho ca 2 window
valid_cutoffs = []
for cutoff in cutoff_candidates:
    _, valid_3m = kiem_tra_va_gan_nhan(df_clean, cutoff, 90, max_date)
    if valid_3m:
        valid_cutoffs.append(cutoff)
print(f"So cutoff HOP LE (du du lieu cho churn 3 thang): {len(valid_cutoffs)}")
print([c.date() for c in valid_cutoffs])


In [ ]:
results = []
for cutoff in cutoff_candidates:
    for window_days, window_name in [(90, '3 thang'), (180, '6 thang')]:
        labels, is_valid = kiem_tra_va_gan_nhan(df_clean, cutoff, window_days, max_date)
        if is_valid:
            results.append({'cutoff': cutoff.date(), 'window': window_name,
                             'n_customers': len(labels), 'churn_rate_%': round(labels.mean() * 100, 2)})
        else:
            results.append({'cutoff': cutoff.date(), 'window': window_name,
                             'n_customers': None, 'churn_rate_%': None})

sensitivity_df = pd.DataFrame(results)
plt.figure(figsize=(11, 5))
sns.lineplot(data=sensitivity_df.dropna(), x='cutoff', y='churn_rate_%', hue='window', marker='o')
plt.title('Ty le churn theo moc cutoff (Sensitivity Analysis)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(sensitivity_df.dropna().groupby('window')['churn_rate_%'].agg(['mean', 'std', 'min', 'max']))


**Can nhin gi**: neu std nho so voi mean, dinh nghia churn on dinh -- danh sach `valid_cutoffs` o tren se dung truc tiep cho Time-based Walk-Forward Validation o Phan 3, KHONG chon ngau nhien 1 cutoff nhu ban truoc nua.

---
# PHAN 2 -- Feature Engineering Nang Cao (da bo sung FrequencyTrend, RepeatPurchaseRatio, Country)

**Diem moi so voi ban truoc**:
- Them `FrequencyTrend` (tuong tu SpendingTrend nhung dem SO DON thay vi tong tien)
- Them `RepeatPurchaseRatio` = (so thang co mua hang) / (so thang ke tu lan mua dau tien den cutoff) -- do MUC DO DEU DAN tham gia, khac voi Frequency (chi dem tong so don)
- Them `Country_grouped` da One-Hot Encode ngay trong ham tra ve, thay vi chi demo rieng

In [ ]:
# Xac dinh top-8 quoc gia CHI dua tren du lieu truoc cutoff SOM NHAT (tranh leakage tu tuong lai)
first_valid_cutoff = valid_cutoffs[0]
obs_for_country = df_clean[df_clean['InvoiceDate'] < first_valid_cutoff]
TOP_COUNTRIES = obs_for_country['Country'].value_counts().head(8).index.tolist()
print("Top 8 quoc gia (dung chung cho moi cutoff, xac dinh tu du lieu som nhat):", TOP_COUNTRIES)


In [ ]:
def tinh_dac_trung_nang_cao(df_raw_input, df_clean_input, cutoff_date, top_countries=TOP_COUNTRIES):
    obs_clean = df_clean_input[df_clean_input['InvoiceDate'] < cutoff_date].copy()
    obs_raw = df_raw_input[df_raw_input['InvoiceDate'] < cutoff_date].copy()

    # --- RFM co ban ---
    rfm = obs_clean.groupby('Customer ID').agg(
        Recency=('InvoiceDate', lambda x: (cutoff_date - x.max()).days),
        Frequency=('Invoice', 'nunique'),
        Monetary=('Revenue', 'sum'),
        AvgOrderValue=('Revenue', 'mean'),
        NumProducts=('StockCode', 'nunique'),
        FirstPurchase=('InvoiceDate', 'min'),
        Country=('Country', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
    ).reset_index()

    # --- Do lech chuan khoang cach mua ---
    def std_khoang_cach(dates):
        d = sorted(dates.unique())
        if len(d) < 3:
            return np.nan
        diffs = np.diff(d) / np.timedelta64(1, 'D')
        return float(np.std(diffs))
    interval_std = (obs_clean.groupby('Customer ID')['InvoiceDate']
                     .apply(std_khoang_cach).rename('IntervalStd').reset_index())

    # --- Xu huong chi tieu VA xu huong tan suat (3 thang gan vs 3 thang truoc) ---
    mid_point = cutoff_date - pd.Timedelta(days=90)
    far_point = cutoff_date - pd.Timedelta(days=180)

    rev_recent = obs_clean[obs_clean['InvoiceDate'] >= mid_point].groupby('Customer ID')['Revenue'].sum()
    rev_prev = obs_clean[(obs_clean['InvoiceDate'] >= far_point) & (obs_clean['InvoiceDate'] < mid_point)].groupby('Customer ID')['Revenue'].sum()
    freq_recent = obs_clean[obs_clean['InvoiceDate'] >= mid_point].groupby('Customer ID')['Invoice'].nunique()
    freq_prev = obs_clean[(obs_clean['InvoiceDate'] >= far_point) & (obs_clean['InvoiceDate'] < mid_point)].groupby('Customer ID')['Invoice'].nunique()

    trend = pd.concat([rev_recent.rename('rev_recent'), rev_prev.rename('rev_prev'),
                        freq_recent.rename('freq_recent'), freq_prev.rename('freq_prev')], axis=1).fillna(0)
    trend['SpendingTrend'] = trend['rev_recent'] - trend['rev_prev']
    trend['FrequencyTrend'] = trend['freq_recent'] - trend['freq_prev']
    trend = trend[['SpendingTrend', 'FrequencyTrend']].reset_index()

    # --- Ty le huy don (can du lieu THO) ---
    total_invoices = obs_raw.groupby('Customer ID')['Invoice'].nunique().rename('TotalInvoices')
    cancelled_invoices = (obs_raw[obs_raw['is_cancelled']].groupby('Customer ID')['Invoice']
                          .nunique().rename('CancelledInvoices'))
    cancel_df = pd.concat([total_invoices, cancelled_invoices], axis=1).fillna(0)
    cancel_df['CancelRate'] = cancel_df['CancelledInvoices'] / cancel_df['TotalInvoices'].replace(0, np.nan)
    cancel_df = cancel_df[['CancelRate']].fillna(0).reset_index()

    # --- Gop tat ca ---
    features = rfm.merge(interval_std, on='Customer ID', how='left')
    features = features.merge(trend, on='Customer ID', how='left')
    features = features.merge(cancel_df, on='Customer ID', how='left')
    features['IntervalStd'] = features['IntervalStd'].fillna(features['IntervalStd'].median())
    features[['SpendingTrend', 'FrequencyTrend', 'CancelRate']] = features[['SpendingTrend', 'FrequencyTrend', 'CancelRate']].fillna(0)

    # --- MOI: RepeatPurchaseRatio = so thang co mua / so thang ke tu lan mua dau ---
    tenure_months = ((cutoff_date - features['FirstPurchase']).dt.days / 30.44).clip(lower=1)
    active_months_series = (obs_clean.assign(ym=obs_clean['InvoiceDate'].dt.to_period('M'))
                             .groupby('Customer ID')['ym'].nunique().rename('ActiveMonths'))
    features = features.merge(active_months_series, on='Customer ID', how='left')
    features['ActiveMonths'] = features['ActiveMonths'].fillna(1)
    features['RepeatPurchaseRatio'] = (features['ActiveMonths'] / tenure_months).clip(upper=1.0)

    # --- MOI: Country_grouped + One-Hot Encoding (dung chung top_countries cho moi cutoff) ---
    features['Country_grouped'] = features['Country'].where(features['Country'].isin(top_countries), 'Other')
    country_dummies = pd.get_dummies(features['Country_grouped'], prefix='Country', drop_first=True)

    features = pd.concat([features.drop(columns=['FirstPurchase', 'Country', 'Country_grouped', 'ActiveMonths']),
                           country_dummies], axis=1)

    return features

# Test thu tren 1 cutoff de xem ket qua
demo_features = tinh_dac_trung_nang_cao(df_raw, df_clean, valid_cutoffs[0])
print(f"So dac trung (bao gom Country dummies): {demo_features.shape[1] - 1}")
demo_features.head()


**Can nhin gi**: `RepeatPurchaseRatio` gan 1.0 nghia la khach mua deu dan hau het cac thang ke tu khi bat dau (rat gan bo); gan 0 nghia la chi mua 1-2 thang roi im bat, du Frequency co the van cao (mua don gan nhau trong thoi gian ngan roi bien mat) -- day la tin hieu KHAC voi Frequency thong thuong.

## Feature Selection (cap nhat voi day du dac trung moi)

In [ ]:
labels_3m, _ = kiem_tra_va_gan_nhan(df_clean, valid_cutoffs[0], 90, max_date)
data_demo = demo_features.merge(labels_3m.rename('churn'), left_on='Customer ID', right_index=True)

feature_cols = [c for c in demo_features.columns if c not in ('Customer ID',)]
print(f"Tong so dac trung dung cho modeling: {len(feature_cols)}")
print(feature_cols)

mi_scores = mutual_info_classif(data_demo[feature_cols], data_demo['churn'], random_state=42)
mi_df = pd.DataFrame({'feature': feature_cols, 'mutual_info': mi_scores}).sort_values('mutual_info', ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x='mutual_info', y='feature', data=mi_df, orient='h')
plt.title('Mutual Information voi nhan churn (cutoff dau tien, minh hoa)')
plt.show()


**Can nhin gi**: neu `FrequencyTrend` va `RepeatPurchaseRatio` co diem MI dang ke (khong gan 0), day la bang chung nen giu lai cac dac trung nay trong mo hinh chinh thuc.

---
# PHAN 3 -- Time-based Walk-Forward Validation (thay the Nested CV ngau nhien)

**Tai sao thay doi**: theo dung yeu cau cua thay -- "chia Train/Validation/Test theo THOI GIAN de tranh data leakage". Voi StratifiedKFold ngau nhien (ban truoc), co the vo tinh dung thong tin gan cutoff MUON hon de du doan cutoff SOM hon trong cung 1 fold. Walk-Forward Validation dam bao: **Train luon o qua khu, Validation o giua, Test luon o tuong lai xa nhat** trong moi fold.

**Cach hoat dong**: voi danh sach `valid_cutoffs` da sap xep theo thoi gian (c1 < c2 < ... < cN), moi "fold" lay 3 cutoff LIEN TIEP: cutoff dau = Train, cutoff giua = Validation (chon hyperparameter), cutoff cuoi = Test (danh gia cuoi cung). Sau do truot toi 3 cutoff tiep theo -- giong ky thuat "walk-forward" pho bien trong du bao chuoi thoi gian.

In [ ]:
def chuan_bi_du_lieu_cho_cutoff(cutoff, churn_window_days=90):
    features = tinh_dac_trung_nang_cao(df_raw, df_clean, cutoff)
    labels, is_valid = kiem_tra_va_gan_nhan(df_clean, cutoff, churn_window_days, max_date)
    if not is_valid:
        return None
    data = features.merge(labels.rename('churn'), left_on='Customer ID', right_index=True)
    return data

# Tao danh sach cac fold walk-forward: (train_cutoff, val_cutoff, test_cutoff)
walk_forward_folds = []
for i in range(len(valid_cutoffs) - 2):
    walk_forward_folds.append((valid_cutoffs[i], valid_cutoffs[i + 1], valid_cutoffs[i + 2]))

print(f"So fold walk-forward: {len(walk_forward_folds)}")
for train_c, val_c, test_c in walk_forward_folds:
    print(f"  Train={train_c.date()} | Val={val_c.date()} | Test={test_c.date()}")


In [ ]:
model_configs = {
    'RandomForest': {
        'builder': lambda params, seed: RandomForestClassifier(random_state=seed, class_weight='balanced', **params),
        'param_grid': [{'n_estimators': 200, 'max_depth': 5}, {'n_estimators': 300, 'max_depth': 8}],
    },
    'XGBoost': {
        'builder': lambda params, seed: xgb.XGBClassifier(random_state=seed, eval_metric='logloss', **params),
        'param_grid': [{'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1},
                       {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05}],
    },
    'LightGBM': {
        'builder': lambda params, seed: lgb.LGBMClassifier(random_state=seed, verbose=-1, **params),
        'param_grid': [{'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1},
                       {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05}],
    },
}


In [ ]:
def walk_forward_evaluate(model_name, config, feature_cols, seeds=(0, 1, 2)):
    """
    Voi moi fold (train/val/test theo thoi gian) va moi seed:
    1. Thu tung bo hyperparameter trong param_grid, fit tren TRAIN, chon bo tot nhat theo ROC-AUC tren VALIDATION.
    2. Dung bo hyperparameter tot nhat do, fit lai tren TRAIN, danh gia CUOI CUNG tren TEST (chua tung dung de chon gi ca).
    """
    fold_results = []
    for fold_idx, (train_c, val_c, test_c) in enumerate(walk_forward_folds):
        train_data = chuan_bi_du_lieu_cho_cutoff(train_c)
        val_data = chuan_bi_du_lieu_cho_cutoff(val_c)
        test_data = chuan_bi_du_lieu_cho_cutoff(test_c)
        if train_data is None or val_data is None or test_data is None:
            continue

        for seed in seeds:
            best_auc, best_params = -1, config['param_grid'][0]
            for params in config['param_grid']:
                model = config['builder'](params, seed)
                model.fit(train_data[feature_cols], train_data['churn'])
                val_prob = model.predict_proba(val_data[feature_cols])[:, 1]
                val_auc = roc_auc_score(val_data['churn'], val_prob)
                if val_auc > best_auc:
                    best_auc, best_params = val_auc, params

            final_model = config['builder'](best_params, seed)
            final_model.fit(train_data[feature_cols], train_data['churn'])
            test_prob = final_model.predict_proba(test_data[feature_cols])[:, 1]

            fold_results.append({
                'model': model_name, 'fold': fold_idx, 'seed': seed,
                'roc_auc': roc_auc_score(test_data['churn'], test_prob),
                'pr_auc': average_precision_score(test_data['churn'], test_prob),
            })
    return pd.DataFrame(fold_results)


In [ ]:
print("Dang chay Walk-Forward Validation cho ca 3 mo hinh (co the mat vai phut)...")
all_results = []
for name, config in model_configs.items():
    print(f"  -> {name}...")
    res = walk_forward_evaluate(name, config, feature_cols)
    all_results.append(res)

results_df = pd.concat(all_results, ignore_index=True)
results_df.head()


In [ ]:
summary = results_df.groupby('model')[['roc_auc', 'pr_auc']].agg(['mean', 'std'])
print(summary)

plt.figure(figsize=(9, 5))
sns.boxplot(x='model', y='roc_auc', data=results_df)
sns.stripplot(x='model', y='roc_auc', data=results_df, color='black', alpha=0.4, jitter=True)
plt.title('ROC-AUC qua cac fold Walk-Forward, theo tung model')
plt.show()

plt.figure(figsize=(9, 5))
sns.boxplot(x='model', y='pr_auc', data=results_df)
sns.stripplot(x='model', y='pr_auc', data=results_df, color='black', alpha=0.4, jitter=True)
plt.title('PR-AUC qua cac fold Walk-Forward, theo tung model')
plt.show()


**Can nhin gi**: PR-AUC thuong THAP HON ROC-AUC ro ret khi du lieu mat can bang lop (dung nhu ly thuyet da neu o Phan 8.2 file kien thuc nen) -- day la ket qua BINH THUONG, khong phai loi. So sanh PR-AUC giua cac model quan trong hon ROC-AUC cho bai toan churn nay.

In [ ]:
# Kiem dinh thong ke giua cac cap model (tren cung fold+seed de so sanh cong bang)
pivot_auc = results_df.pivot_table(index=['fold', 'seed'], columns='model', values='roc_auc')
pivot_prauc = results_df.pivot_table(index=['fold', 'seed'], columns='model', values='pr_auc')

model_names = list(model_configs.keys())
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        m1, m2 = model_names[i], model_names[j]
        stat, p = wilcoxon(pivot_auc[m1], pivot_auc[m2])
        stat_pr, p_pr = wilcoxon(pivot_prauc[m1], pivot_prauc[m2])
        print(f"{m1} vs {m2}: ROC-AUC p={p:.4f} | PR-AUC p={p_pr:.4f}")


**Can nhin gi**: p-value < 0.05 o CA HAI chi so (ROC-AUC va PR-AUC) moi nen ket luan mot mo hinh thuc su tot hon mo hinh kia co y nghia thong ke -- neu chi mot trong hai chi so co y nghia, can than trong khi ket luan trong bai bao.

## So sanh dac trung co ban vs day du (Ablation, cap nhat theo Walk-Forward)

In [ ]:
basic_cols = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'NumProducts']

print("Dang chay Walk-Forward CHI voi dac trung RFM co ban (Random Forest)...")
res_basic = walk_forward_evaluate('RandomForest_basic', model_configs['RandomForest'], basic_cols)

print(f"RFM co ban     -- ROC-AUC: {res_basic['roc_auc'].mean():.3f} +/- {res_basic['roc_auc'].std():.3f} | PR-AUC: {res_basic['pr_auc'].mean():.3f} +/- {res_basic['pr_auc'].std():.3f}")
rf_full = results_df[results_df['model'] == 'RandomForest']
print(f"RFM + nang cao -- ROC-AUC: {rf_full['roc_auc'].mean():.3f} +/- {rf_full['roc_auc'].std():.3f} | PR-AUC: {rf_full['pr_auc'].mean():.3f} +/- {rf_full['pr_auc'].std():.3f}")

pivot_ablation_auc = pd.DataFrame({'basic': res_basic.sort_values(['fold','seed'])['roc_auc'].values,
                                   'full': rf_full.sort_values(['fold','seed'])['roc_auc'].values})
stat, p = wilcoxon(pivot_ablation_auc['basic'], pivot_ablation_auc['full'])
print(f"Wilcoxon (co ban vs day du, ROC-AUC): p-value = {p:.4f}")


---
## Tong ket Phan 1-2-3 (da cap nhat theo 5 diem con thieu)

- [x] Sensitivity analysis nhieu cutoff -- danh sach `valid_cutoffs` dung lai cho Walk-Forward Validation
- [x] `FrequencyTrend` va `RepeatPurchaseRatio` -- da them vao `tinh_dac_trung_nang_cao()`
- [x] `Country_grouped` da One-Hot Encode va MERGE truc tiep vao ma tran dac trung (khong con demo rieng)
- [x] **LightGBM** -- da them lam mo hinh thu 3 (cung Random Forest, XGBoost)
- [x] **Time-based Walk-Forward Validation** -- thay hoan toan StratifiedKFold ngau nhien, Train luon o qua khu so voi Validation va Test
- [x] **PR-AUC** -- da tinh song song voi ROC-AUC o moi fold, dung ca 2 chi so khi kiem dinh Wilcoxon

**Buoc tiep theo**: xem `../01-tong-quan/lo-trinh-theo-de-cuong-giang-vien.md` Buoc 7 (bo sung SHAP dependence plot) va Buoc 8 (xay notebook Business Decision Support).